<a href="https://colab.research.google.com/github/BangachevKiril/RepresentationLearningTheory/blob/main/TorchSigLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

In [ ]:
device ='cuda'

Hugging face torch distribution of siglip: https://huggingface.co/collections/google/siglip-659d5e62f0ae1a57ae0e83ba

## Local Inference on GPU
Model page: https://huggingface.co/google/siglip-base-patch16-224

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/siglip-base-patch16-224)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("zero-shot-image-classification", model="google/siglip-base-patch16-224")
pipe(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png",
    candidate_labels=["animals", "humans", "landscape"],
)

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification

processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
model = AutoModelForZeroShotImageClassification.from_pretrained("google/siglip-base-patch16-224")

In [ ]:
resolution = 224

In [ ]:
from PIL import Image
import requests
import torch
import torchvision.transforms as transforms

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

texts = ["a photo of 2 cats", "a photo of 2 dogs"]
inputs = processor(text=[''], images=image, padding="max_length", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
inputs['pixel_values'].shape

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !wget https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar

In [ ]:
from torchvision import transforms
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
val_transforms = transforms.Compose([
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [ ]:
path = 'drive/MyDrive/imagenetdata/ImageNetValidation/Images/'

In [ ]:
#!wget \https://raw.githubusercontent.com/tensorflow/models/master/research/slim/datasets/imagenet_2012_validation_synset_labels.txt

In [ ]:
# !wget https://raw.githubusercontent.com/tensorflow/tpu/master/tools/datasets/imagenet_to_gcs.py

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Initialize ImageNet dataset
val_dataset = ImageFolder(root=path, transform=val_transforms)

# Optimize DataLoader settings
batch_size = 100
num_workers = 1
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)

In [ ]:
import numpy as np
dim = 768
num = 50000
embeddings = np.zeros((num, dim))

In [ ]:
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        inputs = processor(text=[''], images = batch[0], padding="max_length", return_tensors="pt")
        outputs = model(**inputs, do_rescale=False)
        embeddings[i*batch_size:(i+1)*batch_size, :] = outputs.image_embeds.cpu().numpy()
        print(i)
        if i == 2:
            break
        np.savez('drive/MyDrive/imagenetdata/torch_image_embeddings.npz', embeddings)

In [ ]:
# Preprocess val data

In [ ]:
%ls drive/MyDrive/imagenetdata

In [ ]:
map = {}
with open('drive/MyDrive/imagenetdata/ImageNetValidation/map_clsloc.txt') as f:
    for line in f:
        line_split = line.split(' ')
        id = line_split[0]
        label = " ".join(line_split[-1].split('\n')[0].split('_'))
        map[int(line_split[1])] = label

In [ ]:
import json
with open('drive/MyDrive/imagenetdata/ImageNetValidation/idx_to_name_map.json', 'w') as fp:
    json.dump(map, fp)

In [ ]:
classes= []
map = {}
with open('drive/MyDrive/imagenetdata/ImageNetValidation/Labels.txt') as f:
    for line in f:
        classes.append(line.split('\n')[0])

In [ ]:
classes[:100]

In [ ]:
with open('drive/MyDrive/imagenetdata/ImageNetValidation/val_class_split.txt','w') as fp:
  for i in range(50000):
    fp.write(f"ILSVRC2012_val_000"+"0"*(5 - len(str(i)))+str(i)+".JPEG "+classes[i]+'\n')